### Remote control of oscilloscope RTO 1044
for further SCPI commands check:

https://scdn.rohde-schwarz.com/ur/pws/dl_downloads/pdm/cl_manuals/user_manual/1332_9725_01/RTO_UserManual_en_28.pdf

In [2]:
from RsInstrument import RsInstrument

In [5]:
from RsInstrument import RsInstrument

# Note the different resource string: no ::INSTR, just host:port
resource = "TCPIP::192.168.10.2::5025::SOCKET"

scope = RsInstrument(resource,
                     reset=False,
                     id_query=True,
                     options="SelectVisa=socket")

print(scope.query_str("*IDN?"))
scope.close()


Rohde&Schwarz,RTO,1316.1000k44/400534,3.70.1.0


In [15]:
from RsInstrument import RsInstrument

resource = "TCPIP::192.168.10.2::5025::SOCKET"
scope = RsInstrument(resource,
                     reset=False,
                     id_query=True,
                     options="SelectVisa=socket")

print("Connected to:", scope.query_str("*IDN?"))

# Clear status only; no CHAN or ACQ commands
scope.write_str("*CLS")

# Use whatever channel/trigger/timebase you already set on the front panel.
# Just tell export to use the full waveform:
scope.write_str("EXPort:WAVeform:SCOPe WFM")

n_files = 1          # start with 1 file for testing
acq_per_file = 10    # 10 singles per file
usb_root = "D:"      # check that your USB stick is D: on the scope

for i in range(n_files):
    print(f"Acquiring block {i+1}/{n_files} with {acq_per_file} acquisitions")

    # Let the scope acquire N single shots using its current settings
    for n in range(acq_per_file):
        scope.write_str("SINGle")
        scope.write_str("*WAI")     # wait for this acquisition to finish

    # Build filename, e.g. D:\Run_000.bin
    fname = f"{usb_root}\\Remote_Run_{i:03d}.bin"
    print("Exporting to", fname)

    # Set export file name and format (extension .bin → binary)
    scope.write_str(f"EXPort:WAVeform:NAME '{fname}'")

    # Start the export
    scope.write_str("EXPort:WAVeform:SAVE")

    # Wait until export is complete
    scope.query_str("*OPC?")       # returns "1" when done

print("Done.")
scope.close()


KeyboardInterrupt: 

In [16]:
from RsInstrument import RsInstrument
import numpy as np

scope = RsInstrument("TCPIP::192.168.10.2::5025::SOCKET",
                     reset=False,
                     id_query=True,
                     options="SelectVisa=socket")

# Configure waveform format
scope.write_str("FORMat:DATA REAL,64")
scope.write_str("WAVeform:SOURce CHAN1")
scope.write_str("WAVeform:POINts:MODE RAW")

n_acq = 10_000
batch_size = 1000

for batch_start in range(0, n_acq, batch_size):
    this_batch = min(batch_size, n_acq - batch_start)
    traces = []

    for i in range(this_batch):
        scope.write_str("SINGle")
        scope.write_str("*WAI")

        y = scope.query_bin_or_ascii_real("WAVeform:DATA?")
        y = np.array(y)

        xinc = float(scope.query_str("WAVeform:XINCrement?"))
        xorig = float(scope.query_str("WAVeform:XORigin?"))
        n = len(y)
        x = xorig + xinc * np.arange(n)

        # either:
        # 1) accumulate for saving:
        traces.append(y)
        # or 2) immediately compute jitter/charge here and append just scalars

    traces = np.column_stack(traces)  # (n_samples, this_batch)

    # Save batch compactly or process and discard
    np.savez_compressed(f"/Users/laloscola/ExternalDrive/run_batch_{batch_start:07d}.npz", x=x, y=traces)
    # or call your analyze_run_large(x, traces, ...) here

scope.close()


StatusException: 'TCPIP::192.168.10.2::5025::SOCKET': Instrument error detected: -222,"Data out of range;FORMat:DATA REAL,64"

In [19]:
from RsInstrument import RsInstrument
import time

resource = "TCPIP::192.168.10.2::5025::SOCKET"
scope = RsInstrument(resource,
                     reset=False,
                     id_query=True,
                     options="SelectVisa=socket")

print("Connected to:", scope.query_str("*IDN?"))

# Assume: data logging mode and file settings are already configured on the front panel.
# This script only starts logging, waits for it to finish, and repeats.

n_runs = 1   # number of logging runs you want

for i in range(n_runs):
    print(f"Starting logging run {i+1}/{n_runs}")

    # Start acquisition/logging (equivalent to RUN CONT)
    scope.write_str("RUN")

    # Option A: wait until the scope signals completion via *OPC?
    # (works if the logging operation is treated as an overlapped operation)
    scope.query_str("*OPC?")   # blocks until the operation completes

    # Option B: if *OPC? doesn't reflect logging completion, you can:
    # - sleep for a known duration, or
    # - poll a logging status register defined in the manual.

    print(f"Logging run {i+1} finished")

print("All runs done.")
scope.close()


Connected to: Rohde&Schwarz,RTO,1316.1000k44/400534,3.70.1.0
Starting logging run 1/1
Logging run 1 finished
All runs done.


In [9]:
from RsInstrument import RsInstrument
import time
# ------------- working code to tell the RTO 1044 to take n_files # with acq_per_file # in data logging mode ---------
'''
1. Install RsInstrument on environment -> 'pip install RsInstrument' 
2. connect via LAN cable to RTO
3. on RTO select IP: 192.168.10.2 and local mask: 255.255.255.0
4. on Mac under Network select manually IP:  192.168.10.1 and local mask: 255.255.255.0 
5. select resolution on the RTO under -> horizontal -> setup 
6. select wanted n_files, acq_per_file, usb_root and wait_s -> typically for 1000 acq's at 1ps resolution: wait_s ~ 30s
7. run script
'''
resource = "TCPIP::192.168.10.2::5025::SOCKET"
scope = RsInstrument(resource,
                     reset=False,
                     id_query=True,
                     options="SelectVisa=socket")

print("Connected to:", scope.query_str("*IDN?"))

scope.write_str("*CLS")

n_files = 20         # number of log files
acq_per_file = 5000   # waveforms per file
usb_root = "D:"
wait_s = 150           # rough time estimate per run (adjust as needed)

for i in range(n_files):
    print(f"Configuring data logging run {i+1}/{n_files}")

    scope.write_str(f"ACQuire:COUNt {acq_per_file}")
    scope.write_str("EXPort:WAVeform:DLOGging ON")
    scope.write_str("EXPort:WAVeform:TIMestamps ON")

    fname = f"{usb_root}\\Run_{(i+1):03d}.bin"
    scope.write_str(f"EXPort:WAVeform:NAME '{fname}'")

    print("Starting RUNSingle")
    scope.write_str("RUNSingle")

    # Just wait long enough for acquisition + logging to finish
    # print(f"Sleeping {wait_s} s while logging...")
    time.sleep(wait_s)

    print(f"Finished run {i+1}, file {fname}")

scope.close()


Connected to: Rohde&Schwarz,RTO,1316.1000k44/400534,3.70.1.0
Configuring data logging run 1/20
Starting RUNSingle
Finished run 1, file D:\Run_000.bin
Configuring data logging run 2/20
Starting RUNSingle
Finished run 2, file D:\Run_001.bin
Configuring data logging run 3/20
Starting RUNSingle
Finished run 3, file D:\Run_002.bin
Configuring data logging run 4/20
Starting RUNSingle
Finished run 4, file D:\Run_003.bin
Configuring data logging run 5/20
Starting RUNSingle
Finished run 5, file D:\Run_004.bin
Configuring data logging run 6/20
Starting RUNSingle
Finished run 6, file D:\Run_005.bin
Configuring data logging run 7/20
Starting RUNSingle
Finished run 7, file D:\Run_006.bin
Configuring data logging run 8/20
Starting RUNSingle
Finished run 8, file D:\Run_007.bin
Configuring data logging run 9/20
Starting RUNSingle
Finished run 9, file D:\Run_008.bin
Configuring data logging run 10/20
Starting RUNSingle
Finished run 10, file D:\Run_009.bin
Configuring data logging run 11/20
Starting RUN

In [2]:
from functions_singledata import converter_binary_to_df

In [12]:
converter_binary_to_df('/Users/laloscola/ExternalDrive/Run_002.Wfm.bin')

Found multichannel File, with channels: 1, 2, 3. Using these!


Resolution -> 1e-12
RecordLength -> 20000.0
Number of Acquisitions: 5000
Timestamps recorded: Yes
Number of channels given: 3


,event_id,timestamp,Ch_1,Ch_2
0,0,-8.120000e-09,-0.021702,-0.000752
1,1,-8.120000e-09,-0.022783,-0.001678
2,2,-8.120000e-09,-0.023678,0.000761
3,3,-8.120000e-09,-0.016854,-0.001585
4,4,-8.120000e-09,-0.025346,-0.001400
...,...,...,...,...
99999995,4995,1.187900e-08,-0.016977,0.004436
99999996,4996,1.187900e-08,-0.020003,0.006567
99999997,4997,1.187900e-08,-0.017718,0.001194
99999998,4998,1.187900e-08,-0.018027,-0.004735
